In [3]:
# ── Ячейка 1: инфраструктура ──────────────────────────────────────────────────
import sys; sys.path.insert(0, "..")
import torch, joblib
from src import config, data, preprocess, evaluate, mlp

con = data.get_duckdb_connection()
feature_cols = data.get_feature_cols(con)
pre = preprocess.load_preprocessor()
print(f"признаков raw={len(feature_cols)}  |  linear-view={pre.n_features_out_}")

# pos_weight для дисбаланса = n_neg / n_pos (как scale_pos_weight у XGBoost).
counts = con.execute(
    f"SELECT label_binary, COUNT(*) FROM read_parquet('{config.TRAIN_PATH}') "
    "GROUP BY label_binary ORDER BY label_binary"
).fetchall()
pos_weight = counts[0][1] / counts[1][1]
print(f"pos_weight = {pos_weight:.4f}  |  torch device = cpu")

признаков raw=76  |  linear-view=80
pos_weight = 2.7091  |  torch device = cpu


In [5]:
# ── Ячейка 2: обучение MLP (early stopping по val) ────────────────────────────
model = mlp.train_mlp(
    pre, feature_cols,
    epochs=30, lr=1e-3, patience=4,
    pos_weight=pos_weight, device="cpu",
)
torch.save(model.state_dict(), config.MODELS_DIR / "mlp.pt")
print(f"Модель сохранена → {config.MODELS_DIR / 'mlp.pt'}")

KeyboardInterrupt: 

In [6]:
# ── Ячейка 3: оценка на VAL (подбор порога) ───────────────────────────────────
predict_mlp = mlp.make_predict_fn(model, pre, device="cpu")

m_val = evaluate.evaluate(
    "mlp", predict_mlp, config.VAL_PATH, feature_cols,
    threshold=None, split_name="val",
)
best_thr = m_val["threshold"]
print(f"\nПодобранный на val порог: {best_thr}")


  mlp  (val)
строк 2,053,688 за 6.44s  |  порог 0.9175 (tuned_on_this_split)
F1_macro 0.9983  MCC 0.9967  PR-AUC 0.99975  ROC-AUC 0.999917  FPR@95 0.000268
TN=1,497,821  FP=2,179  FN=520  TP=553,168  (FPR=0.0015, FNR=0.0009)

per-class recall:
  Benign                           n=1,500,000  recall=99.85%
  DoS Hulk                         n= 270,445  recall=100.00%
  DDoS HOIC                        n= 161,157  recall=99.99%
  DDoS LOIC-HTTP                   n=  43,399  recall=100.00%
  FTP-Patator                      n=  28,545  recall=100.00%
  DoS Slowhttptest                 n=  15,832  recall=100.00%
  Bot                              n=  14,423  recall=99.69%
  SSH-Patator                      n=  13,897  recall=100.00%
  DoS GoldenEye                    n=   4,029  recall=99.18%
  DoS Slowloris                    n=   1,541  recall=75.67%
  DDoS LOIC-UDP                    n=     357  recall=97.76%
  Web Attack - Brute Force         n=      39  recall=41.03% ◄
  Web Attack - 

In [7]:
# ── Ячейка 4: оценка на TEST (замороженный порог) ─────────────────────────────
m_test = evaluate.evaluate(
    "mlp", predict_mlp, config.TEST_PATH, feature_cols,
    threshold=best_thr, split_name="test",
)


  mlp  (test)
строк 2,053,697 за 6.17s  |  порог 0.9175 (frozen_from_val)
F1_macro 0.9981  MCC 0.9961  PR-AUC 0.996667  ROC-AUC 0.999596  FPR@95 0.000609
TN=1,497,339  FP=2,661  FN=464  TP=553,233  (FPR=0.0018, FNR=0.0008)

per-class recall:
  Benign                           n=1,500,000  recall=99.82%
  DoS Hulk                         n= 270,445  recall=100.00%
  DDoS HOIC                        n= 161,157  recall=99.99%
  DDoS LOIC-HTTP                   n=  43,400  recall=100.00%
  FTP-Patator                      n=  28,545  recall=100.00%
  DoS Slowhttptest                 n=  15,833  recall=100.00%
  Bot                              n=  14,424  recall=99.69%
  SSH-Patator                      n=  13,898  recall=100.00%
  DoS GoldenEye                    n=   4,030  recall=99.03%
  DoS Slowloris                    n=   1,542  recall=80.42%
  DDoS LOIC-UDP                    n=     358  recall=94.69%
  Web Attack - Brute Force         n=      39  recall=41.03% ◄
  Web Attack - XS